# Platelet Policy Lab

**Goal:** find a platelet stock rule that wastes less AND leaves fewer patients without platelets.

**How it works**
- `bb_daily.csv` is **last year** (Oct 2025 – Sep 2026). You learn from it.
- `bb_sim.py` simulates **next year** (Oct 2026 – Sep 2027) with new randomness. You test your rules on it.
- Every rule faces exactly the same patients and donors, so differences come from your rule alone.

**Your three levers**
1. `platelet_target(bank, today, recent_demand)` → how many units the bank aims to hold
2. `capacity_plan(bank, today)` → extra platelet units it can prepare per day (donor drives, extra shifts)
3. `transfers=True` → near-expiry units move to banks that are short

**Two numbers to watch:** `unmet_%` (patients not served) and `waste_%` (units discarded).

Keep this notebook in the same folder as `bb_sim.py` and `bb_daily.csv`. Run cells top to bottom (Shift + Enter).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from bb_sim import simulate, scorecard, USUAL_DEMAND, BANKS

pd.set_option("display.width", 200)
results = []   # every rule you test gets added here for comparison

## Step 1 — Look at last year's platelet demand

Before writing any rule, look at the pattern. What happens around August–October?

In [ ]:
hist = pd.read_csv("bb_daily.csv", parse_dates=["date"])
plt_hist = (hist[hist.component == "Platelets"]
            .groupby(["bank", "date"])["demand"].sum().reset_index())
plt_hist["month"] = plt_hist["date"].dt.month

monthly_avg = plt_hist.groupby(["month", "bank"])["demand"].mean().unstack()
month_order = [10, 11, 12, 1, 2, 3, 4, 5, 6, 7, 8, 9]   # Oct -> Sep
monthly_avg.loc[month_order].plot(marker="o", figsize=(9, 4),
                                  title="Average daily platelet demand by month (last year)")
plt.xticks(range(12), ["Oct","Nov","Dec","Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep"])
plt.ylabel("Units per day"); plt.xlabel(""); plt.show()

monthly_avg.loc[month_order].round(1)

## Step 2 — Baseline: what banks do today

Each bank keeps **3 days of its usual demand**, all year round. This is the number to beat.

In [ ]:
baseline = simulate()          # uses the current fixed rule
results.append(scorecard("1. Current fixed rule", baseline))
baseline["summary"]

## Step 3 — Rule based on recent demand

Idea: instead of a fixed number, look at the **last 7 days** of demand.
`recent_demand` is a list of daily demand so far, most recent last.

In [ ]:
def recent_rule(bank, today, recent_demand):
    if not recent_demand:                      # first day: no history yet
        return round(USUAL_DEMAND[bank] * 3)
    last_week = np.mean(recent_demand[-7:])
    return round(last_week * 3)               # 3 days of cover

r = simulate(platelet_target=recent_rule)
results.append(scorecard("2. Last 7 days x 3", r))
r["summary"]

**Question:** did it help much? Why might reacting to *last week* be too slow when dengue season starts?

## Step 4 — Rule based on a seasonal forecast

Use last year's data to forecast: *"In this month, this bank usually needs X units a day."*

`target = forecast × COVER_DAYS + BUFFER`

Try changing `COVER_DAYS` and `BUFFER` and re-running.

In [ ]:
# forecast: average daily demand per bank per month, from last year
forecast = plt_hist.groupby(["bank", "month"])["demand"].mean().to_dict()

COVER_DAYS = 2.5   # days of demand to keep in stock
BUFFER = 3         # extra safety units

def seasonal_rule(bank, today, recent_demand):
    expected = forecast[(bank, today.month)]
    return round(expected * COVER_DAYS + BUFFER)

r = simulate(platelet_target=seasonal_rule)
results.append(scorecard(f"3. Seasonal x {COVER_DAYS} + {BUFFER}", r))
r["summary"]

**Question:** waste and shortages may not move much. Look at Bank A in September in `r["monthly"]` —
is the problem the *target*, or the bank's ability to *prepare* enough platelets? (Hint: Bank A can prepare ~30 units/day.)

## Step 5 — Add capacity before the peak

The forecast tells you weeks ahead that Aug–Oct will be busy. That's time to arrange donor drives,
single-donor (apheresis) platelets and extra separation shifts.

In [ ]:
PEAK_MONTHS = [8, 9, 10]
EXTRA_UNITS = 15        # extra platelet units per day each bank can prepare in peak months

def peak_capacity(bank, today):
    return EXTRA_UNITS if today.month in PEAK_MONTHS else 0

r = simulate(platelet_target=seasonal_rule, capacity_plan=peak_capacity)
results.append(scorecard("4. Seasonal + peak capacity", r))
r["summary"]

## Step 6 — Share between banks

Each morning, platelets on their **last usable day** that a bank won't use are sent to the bank
that is furthest below its target.

In [ ]:
r = simulate(platelet_target=seasonal_rule, capacity_plan=peak_capacity, transfers=True)
results.append(scorecard("5. Seasonal + capacity + transfers", r))
print("Units transferred:", r["transfers"])
r["summary"]

## Step 7 — Compare every rule you tried

In [ ]:
pd.DataFrame(results)

## Step 8 — The trade-off: you can't get both to zero

More cover days → fewer shortages, more waste. Fewer cover days → less waste, more shortages.
This chart shows every option, so the blood bank officer can **choose the balance**.

In [ ]:
curve = []
for cover in [1.0, 1.5, 2.0, 2.5, 3.0, 3.5]:
    def rule(bank, today, recent_demand, c=cover):
        return round(forecast[(bank, today.month)] * c + BUFFER)
    s = scorecard(f"cover {cover}", simulate(rule, peak_capacity, transfers=True))
    curve.append({"cover_days": cover, "unmet_%": s["unmet_%"], "waste_%": s["waste_%"]})
curve = pd.DataFrame(curve)

base = results[0]
plt.figure(figsize=(7, 4.5))
plt.plot(curve["waste_%"], curve["unmet_%"], marker="o", label="Seasonal + capacity + transfers")
for _, row in curve.iterrows():
    plt.annotate(f'{row.cover_days}d', (row["waste_%"], row["unmet_%"]),
                 textcoords="offset points", xytext=(5, 5))
plt.scatter([base["waste_%"]], [base["unmet_%"]], color="red", s=80, label="Current fixed rule")
plt.xlabel("Waste % (units discarded)"); plt.ylabel("Unmet demand %")
plt.title("Waste vs shortage: every option")
plt.legend(); plt.grid(alpha=0.3); plt.show()
curve

## Your turn

- [ ] Which cover-days setting would you recommend, and why? (A shortage can delay a cancer patient's chemotherapy.)
- [ ] Try a different `BUFFER` for Bank A only (hint: `if bank.startswith("Bank A")`).
- [ ] Try a different forecast: weekday vs weekend demand, or a time-series model (SARIMA/Prophet).
- [ ] Which single lever gave the biggest improvement: forecast, capacity or transfers?
- [ ] Is a 15-unit capacity boost realistic? What would it take operationally?

**Write your impact statement** (for the product brief and interviews), using your own numbers:

> "Against the current fixed rule, a seasonal forecast with peak-season capacity planning and
> near-expiry transfers cut unmet platelet demand from __% to __% and waste from __% to __%,
> in a simulation calibrated to published Indian blood bank data."

Say *simulation*, not *real-world result*. That's what makes it interview-defensible.